# Evaluation & Inference

This notebook loads the best model checkpoint from training,
visualises learning curves, generates a confusion matrix,
and runs inference on the unseen test set.

In [ ]:
import pickle
from pathlib import Path
import tensorflow as tf
from src.pipeline.evaluation import evaluate_model, plot_training_history
from src.pipeline.data_pipeline import get_train_val_test

BATCH_SIZE = 32
IMG_SIZE = (150, 150)

# Load test data only
_, _, test_ds = get_train_val_test(
    data_dir="../data/processed",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

# Extract class names from directory structure
class_names = sorted([
    p.name for p in Path("../data/processed/train").iterdir()
    if p.is_dir()
])
print(f"Classes: {class_names}")

In [ ]:
# Load the best checkpoint from training
CHECKPOINT_PATH = "../models_output/checkpoints/best.keras"

if Path(CHECKPOINT_PATH).exists():
    model = tf.keras.models.load_model(CHECKPOINT_PATH)
    print(f"Loaded checkpoint: {CHECKPOINT_PATH}")
    model.summary()
else:
    print("No checkpoint found. Train the model first via 03_model_training.ipynb")
    # Fallback: build untrained model for structural demo
    from src.pipeline.model import build_cnn_model, compile_model
    model = build_cnn_model(input_shape=(*IMG_SIZE, 3), num_classes=len(class_names))
    compile_model(model)
    print("Using untrained model (demo mode).")

In [ ]:
# Plot training curves from persisted history
hist_path = Path("../reports/figures/training_history.pkl")
if hist_path.exists():
    with open(hist_path, "rb") as f:
        history_data = pickle.load(f)
    class FakeHistory:
        pass
    fake_hist = FakeHistory()
    fake_hist.history = history_data
    plot_training_history(fake_hist, save_dir="../reports/figures")
    print("Training curves saved to reports/figures/accuracy_loss_curves.png")
else:
    print("No training history found. Run 03_model_training.ipynb first.")

In [ ]:
# Full evaluation on test set
results = evaluate_model(
    model,
    test_ds,
    class_names=class_names,
    save_dir="../reports/figures",
)

print(f"\n{'='*40}")
print(f"Test Accuracy: {results['accuracy']:.4f}")
print(f"{'='*40}")
print(f"Confusion matrix saved to ../reports/figures/confusion_matrix.png")
print(f"Classification report saved to ../reports/figures/classification_report.txt")

In [ ]:
# Optional: export all deployment artifacts
from src.pipeline.export_utils import export_all

sizes = export_all(
    model,
    class_names=class_names,
    saved_model_path="../models_output/saved_model",
    tflite_path="../models_output/tflite/model.tflite",
    tfjs_path="../models_output/tfjs",
)
print(f"\nArtifact sizes: {sizes}")